# H3K4me3 TF Motif Analysis


## Setup


In [19]:
import re
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
BASE_DIR       = Path("/coh_labs/mvandenbrink/users/pkaur/6_tff1/3_cutnrun_relaxed/6_homer")
MOTIF_DIR      = BASE_DIR / "motifs"
MOTIF_GENE_DIR = BASE_DIR / "motif_gene_links"
OUTPUT_DIR     = BASE_DIR / "analysis_output"

# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
SAMPLES = [
    "GFP_H3K4me3.seacr.consensus.peak_counts",
    "TFF1_H3K4me3.seacr.consensus.peak_counts",
]
SAMPLE_LABELS = {s: s.split("_")[0] for s in SAMPLES}   # "GFP", "TFF1"

# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
Q_THRESH       = 0.05          # Benjamini q-value significance cutoff
PROMOTER_LABEL = "promoter-TSS" # HOMER annotation label for −1 kb to +100 bp
ANNOT_COLS     = 21            # number of fixed annotation columns (PeakID … GC%)

TF_FAMILIES = {
    "Pax":  r"(?i)pax",
    "AP1":  r"(?i)(fosb|junb|fos|jun|ap-1)",
    "p53":  r"(?i)(p53|tp53)",
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR)

Output directory: /coh_labs/mvandenbrink/users/pkaur/6_tff1/3_cutnrun_relaxed/6_homer/analysis_output


## 1. Load motif enrichment


In [20]:
def load_known_results(sample_id: str, sample_label: str) -> pd.DataFrame:
    path = MOTIF_DIR / sample_id / "knownResults.txt"
    df = pd.read_csv(path, sep="\t")

    pct_cols = [
        "% of Target Sequences with Motif",
        "% of Background Sequences with Motif",
    ]
    for col in pct_cols:
        df[col] = df[col].astype(str).str.replace("%", "", regex=False).astype(float)

    df = df.rename(columns={
        "q-value (Benjamini)":                   "q-value",
        "% of Target Sequences with Motif":      "% Target",
        "% of Background Sequences with Motif":  "% Background",
    })

    df["Sample"] = sample_label
    bg = df["% Background"].replace(0, float("nan"))
    df["Fold Enrichment"] = (df["% Target"] / bg).round(2)
    return df


frames = []
for s in SAMPLES:
    label = SAMPLE_LABELS[s]
    df = load_known_results(s, label)
    frames.append(df)
    sig_count = (df["q-value"] <= Q_THRESH).sum()
    print(f"{label}: {len(df)} total motifs, {sig_count} significant (q ≤ {Q_THRESH})")

enrich_df = pd.concat(frames, ignore_index=True)
sig_df    = enrich_df[enrich_df["q-value"] <= Q_THRESH].copy()

GFP: 472 total motifs, 85 significant (q ≤ 0.05)
TFF1: 472 total motifs, 20 significant (q ≤ 0.05)


## 2. Top enriched motifs


In [21]:
display_cols = [
    "Motif Name", "Consensus", "Log P-value", "q-value",
    "% Target", "% Background", "Fold Enrichment",
]

for label in SAMPLE_LABELS.values():
    print(f"\n{'='*70}")
    print(f"  {label} — Top 20 enriched motifs")
    print(f"{'='*70}")
    sub = (
        sig_df[sig_df["Sample"] == label]
        .sort_values("Log P-value")
        .head(20)[display_cols]
        .reset_index(drop=True)
    )
    display(sub)


  GFP — Top 20 enriched motifs


,Motif Name,Consensus,Log P-value,q-value,% Target,% Background,Fold Enrichment
0,IRF1(IRF)/PBMC-IRF1-ChIP-Seq(GSE43036)/Homer,GAAAGTGAAAGT,-17.300,0.0000,1.35,0.90,1.50
1,Tlx?(NR)/NPC-H3K4me1-ChIP-Seq(GSE16256)/Homer,CTGGCAGSCTGCCA,-16.240,0.0000,4.33,3.53,1.23
2,Smad3(MAD)/NPC-Smad3-ChIP-Seq(GSE36673)/Homer,TWGTCTGV,-13.810,0.0002,28.44,26.74,1.06
3,IRF8(IRF)/BMDM-IRF8-ChIP-Seq(GSE77884)/Homer,GRAASTGAAAST,-12.020,0.0007,2.94,2.39,1.23
4,Tbx5(T-box)/HL1-Tbx5.biotin-ChIP-Seq(GSE21529)...,AGGTGTCA,-11.750,0.0007,30.63,29.05,1.05
5,BMYB(HTH)/Hela-BMYB-ChIP-Seq(GSE27030)/Homer,NHAACBGYYV,-11.710,0.0007,13.88,12.72,1.09
6,Six2(Homeobox)/NephronProgenitor-Six2-ChIP-Seq...,GWAAYHTGAKMC,-11.410,0.0007,7.81,6.93,1.13
7,NF1-halfsite(CTF)/LNCaP-NF1-ChIP-Seq(Unpublish...,YTGCCAAG,-10.590,0.0015,17.65,16.43,1.07
8,Tgif2(Homeobox)/mES-Tgif2-ChIP-Seq(GSE55404)/H...,TGTCANYT,-10.500,0.0015,28.81,27.36,1.05
9,Nkx2.2(Homeobox)/NPC-Nkx2.2-ChIP-Seq(GSE61673)...,BTBRAGTGSN,-9.880,0.0024,18.47,17.28,1.07



  TFF1 — Top 20 enriched motifs


,Motif Name,Consensus,Log P-value,q-value,% Target,% Background,Fold Enrichment
0,Npas4(bHLH)/Neuron-Npas4-ChIP-Seq(GSE127793)/H...,NHRTCACGACDN,-13.890,0.0004,6.91,6.22,1.11
1,ZNF416(Zf)/HEK293-ZNF416.GFP-ChIP-Seq(GSE58341...,WDNCTGGGCA,-12.920,0.0006,14.17,13.25,1.07
2,p73(p53)/Trachea-p73-ChIP-Seq(PRJNA310161)/Homer,NRRRCAWGTCCDGRCATGYY,-12.210,0.0008,0.49,0.33,1.48
3,Smad3(MAD)/NPC-Smad3-ChIP-Seq(GSE36673)/Homer,TWGTCTGV,-11.740,0.0009,30.14,28.98,1.04
4,p53(p53)/Saos-p53-ChIP-Seq(GSE15780)/Homer,RRCATGYCYRGRCATGYYYN,-10.920,0.0014,0.84,0.64,1.31
5,p53(p53)/Saos-p53-ChIP-Seq/Homer,RRCATGYCYRGRCATGYYYN,-10.920,0.0014,0.84,0.64,1.31
6,PR(NR)/T47D-PR-ChIP-Seq(GSE31130)/Homer,VAGRACAKNCTGTBC,-9.445,0.0053,20.35,19.46,1.05
7,EBF1(EBF)/Near-E2A-ChIP-Seq(GSE21512)/Homer,GTCCCCWGGGGA,-9.370,0.0053,11.87,11.17,1.06
8,GLI3(Zf)/Limb-GLI3-ChIP-Chip(GSE11077)/Homer,CGTGGGTGGTCC,-8.851,0.0075,1.34,1.11,1.21
9,Tgif2(Homeobox)/mES-Tgif2-ChIP-Seq(GSE55404)/H...,TGTCANYT,-8.741,0.0075,31.73,30.75,1.03


## 3. TF-family enrichment


In [22]:
enrich_df["Significant"] = enrich_df["q-value"] <= Q_THRESH

fam_display_cols = [
    "Sample", "Motif Name", "Log P-value", "q-value",
    "Fold Enrichment", "Significant",
]

for fam, regex in TF_FAMILIES.items():
    mask = enrich_df["Motif Name"].str.contains(regex, regex=True, na=False)
    sub  = enrich_df[mask][fam_display_cols].reset_index(drop=True)
    print(f"\n{'='*70}")
    print(f"  {fam} — {len(sub)} motif rows across both samples")
    print(f"{'='*70}")
    if sub.empty:
        print("  No motifs matched this family.")
    else:
        display(sub)
    print("  Note: non-significant motifs were still scanned in the motif-gene linking step.")


  Pax — 16 motif rows across both samples


,Sample,Motif Name,Log P-value,q-value,Fold Enrichment,Significant
0,GFP,"Pax8(Paired,Homeobox)/Thyroid-Pax8-ChIP-Seq(GS...",-5.568000,0.0324,1.13,True
1,GFP,"PAX5(Paired,Homeobox)/GM12878-PAX5-ChIP-Seq(GS...",-4.661000,0.0508,1.10,False
2,GFP,"PAX6(Paired,Homeobox)/Forebrain-Pax6-ChIP-Seq(...",-1.377000,0.4512,1.08,False
3,GFP,"PAX5(Paired,Homeobox),condensed/GM12878-PAX5-C...",-0.798900,0.6827,1.01,False
4,GFP,"PAX3:FKHR-fusion(Paired,Homeobox)/Rh4-PAX3:FKH...",-0.452000,0.8184,0.98,False
5,GFP,"Pax7(Paired,Homeobox),long/Myoblast-Pax7-ChIP-...",-0.296800,0.8926,0.88,False
6,GFP,"Pax7(Paired,Homeobox),longest/Myoblast-Pax7-Ch...",-0.005998,1.0000,0.40,False
7,GFP,"Pax7(Paired,Homeobox)/Myoblast-Pax7-ChIP-Seq(G...",-0.001280,1.0000,0.67,False
8,TFF1,"Pax8(Paired,Homeobox)/Thyroid-Pax8-ChIP-Seq(GS...",-7.183000,0.0224,1.11,True
9,TFF1,"PAX5(Paired,Homeobox),condensed/GM12878-PAX5-C...",-2.032000,0.4073,1.08,False


  Note: non-significant motifs were still scanned in the motif-gene linking step.

  AP1 — 14 motif rows across both samples


/tmp/ipykernel_1041327/3900697180.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = enrich_df["Motif Name"].str.contains(regex, regex=True, na=False)


,Sample,Motif Name,Log P-value,q-value,Fold Enrichment,Significant
0,GFP,Fosl2(bZIP)/3T3L1-Fosl2-ChIP-Seq(GSE56872)/Homer,-9.2370,0.0027,1.22,True
1,GFP,Jun-AP1(bZIP)/K562-cJun-ChIP-Seq(GSE31477)/Homer,-8.0930,0.0063,1.24,True
2,GFP,Fos(bZIP)/TSC-Fos-ChIP-Seq(GSE110950)/Homer,-6.6640,0.0163,1.13,True
3,GFP,JunB(bZIP)/DendriticCells-Junb-ChIP-Seq(GSE360...,-4.9150,0.0433,1.11,True
4,GFP,AP-1(bZIP)/ThioMac-PU.1-ChIP-Seq(GSE21512)/Homer,-4.4620,0.0560,1.08,False
5,GFP,c-Jun-CRE(bZIP)/K562-cJun-ChIP-Seq(GSE31477)/H...,-0.3464,0.8659,0.97,False
6,GFP,JunD(bZIP)/K562-JunD-ChIP-Seq/Homer,-0.2210,0.9227,0.93,False
7,TFF1,Fosl2(bZIP)/3T3L1-Fosl2-ChIP-Seq(GSE56872)/Homer,-7.6970,0.0143,1.13,True
8,TFF1,Jun-AP1(bZIP)/K562-cJun-ChIP-Seq(GSE31477)/Homer,-6.9900,0.0256,1.14,True
9,TFF1,AP-1(bZIP)/ThioMac-PU.1-ChIP-Seq(GSE21512)/Homer,-4.7890,0.0960,1.06,False


  Note: non-significant motifs were still scanned in the motif-gene linking step.

  p53 — 10 motif rows across both samples


/tmp/ipykernel_1041327/3900697180.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = enrich_df["Motif Name"].str.contains(regex, regex=True, na=False)


,Sample,Motif Name,Log P-value,q-value,Fold Enrichment,Significant
0,GFP,p63(p53)/Keratinocyte-p63-ChIP-Seq(GSE17611)/H...,-8.705,0.0039,1.19,True
1,GFP,p53(p53)/Saos-p53-ChIP-Seq(GSE15780)/Homer,-5.268,0.0364,1.30,True
2,GFP,p53(p53)/Saos-p53-ChIP-Seq/Homer,-5.268,0.0364,1.30,True
3,GFP,p73(p53)/Trachea-p73-ChIP-Seq(PRJNA310161)/Homer,-5.260,0.0364,1.41,True
4,GFP,p53(p53)/mES-cMyc-ChIP-Seq(GSE11431)/Homer,-2.367,0.2432,1.27,False
5,TFF1,p73(p53)/Trachea-p73-ChIP-Seq(PRJNA310161)/Homer,-12.210,0.0008,1.48,True
6,TFF1,p53(p53)/Saos-p53-ChIP-Seq(GSE15780)/Homer,-10.920,0.0014,1.31,True
7,TFF1,p53(p53)/Saos-p53-ChIP-Seq/Homer,-10.920,0.0014,1.31,True
8,TFF1,p63(p53)/Keratinocyte-p63-ChIP-Seq(GSE17611)/H...,-7.879,0.0128,1.12,True
9,TFF1,p53(p53)/mES-cMyc-ChIP-Seq(GSE11431)/Homer,-1.095,0.6934,1.09,False


  Note: non-significant motifs were still scanned in the motif-gene linking step.


## 4. Load motif-gene links


In [23]:
def load_motif_gene(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    df = df.rename(columns={df.columns[0]: "PeakID"})
    return df


mg = {}
for s in SAMPLES:
    label = SAMPLE_LABELS[s]
    path  = MOTIF_GENE_DIR / f"{s}_known_motif_genes.txt"
    df    = load_motif_gene(path)
    mg[label] = df
    n_motif_cols = len(df.columns[ANNOT_COLS:])
    print(f"{label}: {len(df):,} peaks, {n_motif_cols} motif columns")

GFP: 16,924 peaks, 91 motif columns
TFF1: 33,179 peaks, 47 motif columns


## 5. TF-family motif columns


In [24]:
# tf_cols[family][label] = list of matching column names
tf_cols = {fam: {} for fam in TF_FAMILIES}

for fam, regex in TF_FAMILIES.items():
    print(f"\n=== {fam} ===")
    for label, df in mg.items():
        all_motif_cols = df.columns[ANNOT_COLS:].tolist()
        cols  = [c for c in all_motif_cols if re.search(regex, c)]
        short = [c.split("/")[0] for c in cols]
        tf_cols[fam][label] = cols
        print(f"  {label}: {len(cols)} column(s)")
        for sn in short:
            print(f"    - {sn}")


=== Pax ===
  GFP: 2 column(s)
    - Pax8(Paired,Homeobox)
    - PAX5(Paired,Homeobox)
  TFF1: 1 column(s)
    - Pax8(Paired,Homeobox)

=== AP1 ===
  GFP: 4 column(s)
    - Fosl2(bZIP)
    - Jun-AP1(bZIP)
    - Fos(bZIP)
    - JunB(bZIP)
  TFF1: 3 column(s)
    - Fosl2(bZIP)
    - Jun-AP1(bZIP)
    - AP-1(bZIP)

=== p53 ===
  GFP: 4 column(s)
    - p63(p53)
    - p53(p53)
    - p53(p53)
    - p73(p53)
  TFF1: 4 column(s)
    - p63(p53)
    - p73(p53)
    - p53(p53)
    - p53(p53)


## 6. Promoter-proximal peaks


In [25]:
def extract_tf_promoter_genes(
    df: pd.DataFrame,
    col_list: list,
    sample_label: str,
    promoter_label: str = PROMOTER_LABEL,
) -> pd.DataFrame:
    """Return tidy DataFrame of promoter peaks with ≥1 hit in any col_list column."""
    if not col_list:
        return pd.DataFrame()

    promo = df[df["Annotation"].str.contains(promoter_label, na=False)].copy()
    hit_mask = promo[col_list].gt(0).any(axis=1)
    promo = promo[hit_mask].copy()

    id_cols = [
        "PeakID", "Chr", "Start", "End", "Distance to TSS",
        "Gene Name", "Gene Description", "Gene Type",
    ]
    long = promo[id_cols + col_list].melt(
        id_vars=id_cols,
        value_vars=col_list,
        var_name="Motif_full",
        value_name="Motif Hits",
    )
    long = long[long["Motif Hits"] > 0].copy()
    long["Sample"] = sample_label
    long["Motif"]  = long["Motif_full"].str.split("/").str[0]

    return long[[
        "Sample", "Gene Name", "Gene Description", "Gene Type",
        "Motif", "Motif Hits", "Distance to TSS", "Chr", "Start", "End",
    ]].reset_index(drop=True)

In [26]:
# tf_gene_tables[family][label] = tidy DataFrame
tf_gene_tables = {}

for fam in TF_FAMILIES:
    print(f"\n{'='*70}")
    print(f"  {fam}")
    print(f"{'='*70}")
    tf_gene_tables[fam] = {}

    for label, df in mg.items():
        cols = tf_cols[fam][label]
        tbl  = extract_tf_promoter_genes(df, cols, label)
        tf_gene_tables[fam][label] = tbl

        if tbl.empty:
            print(f"\n{label}: no promoter-TSS peaks with {fam} motif hits")
        else:
            print(f"\n{label}: {len(tbl)} peak×motif rows | {tbl['Gene Name'].nunique()} unique genes")
            display(tbl)


  Pax

GFP: 3848 peak×motif rows | 2391 unique genes


,Sample,Gene Name,Gene Description,Gene Type,Motif,Motif Hits,Distance to TSS,Chr,Start,End
0,GFP,Cyp24a1,"cytochrome P450, family 24, subfamily a, polyp...",protein-coding,"Pax8(Paired,Homeobox)",1,-655,chr2,170496651,170498950
1,GFP,Tox2,TOX high mobility group box family member 2,protein-coding,"Pax8(Paired,Homeobox)",1,-277,chr2,163200851,163204850
2,GFP,Bpifb1,"BPI fold containing family B, member 1",protein-coding,"Pax8(Paired,Homeobox)",2,95,chr2,154197951,154203000
3,GFP,Tas1r2,"taste receptor, type 1, member 2",protein-coding,"Pax8(Paired,Homeobox)",6,-863,chr4,139647551,139657800
4,GFP,4933406J09Rik,RIKEN cDNA 4933406J09 gene,ncRNA,"Pax8(Paired,Homeobox)",1,-10,chr6,134402501,134404350
...,...,...,...,...,...,...,...,...,...,...
3843,GFP,Nle1,notchless homolog 1,protein-coding,"PAX5(Paired,Homeobox)",1,70,chr11,82905651,82911000
3844,GFP,Zc3h12c,zinc finger CCCH type containing 12C,protein-coding,"PAX5(Paired,Homeobox)",1,-419,chr9,52164351,52173300
3845,GFP,Slc4a3,"solute carrier family 4 (anion exchanger), mem...",protein-coding,"PAX5(Paired,Homeobox)",2,38,chr1,75544651,75548350
3846,GFP,Thbd,thrombomodulin,protein-coding,"PAX5(Paired,Homeobox)",3,63,chr2,148405651,148410600



TFF1: 1929 peak×motif rows | 1915 unique genes


,Sample,Gene Name,Gene Description,Gene Type,Motif,Motif Hits,Distance to TSS,Chr,Start,End
0,TFF1,Sycp2,synaptonemal complex protein 2,protein-coding,"Pax8(Paired,Homeobox)",1,-692,chr2,178406251,178410450
1,TFF1,Ctcfl,CCCTC-binding factor like,protein-coding,"Pax8(Paired,Homeobox)",2,-900,chr2,173118101,173122750
2,TFF1,Gm13133,predicted gene 13133,ncRNA,"Pax8(Paired,Homeobox)",2,-907,chr4,154546501,154552250
3,TFF1,St6galnac3,"ST6 (alpha-N-acetyl-neuraminyl-2,3-beta-galact...",protein-coding,"Pax8(Paired,Homeobox)",1,-917,chr3,153724451,153727650
4,TFF1,Xkr7,X-linked Kx blood group related 7,protein-coding,"Pax8(Paired,Homeobox)",1,-702,chr2,153028351,153033950
...,...,...,...,...,...,...,...,...,...,...
1924,TFF1,Ptgs2os,"prostaglandin-endoperoxide synthase 2, opposit...",ncRNA,"Pax8(Paired,Homeobox)",2,199,chr1,150097101,150102250
1925,TFF1,Serp2,stress-associated endoplasmic reticulum protei...,protein-coding,"Pax8(Paired,Homeobox)",1,-834,chr14,76554801,76560700
1926,TFF1,Lyst,lysosomal trafficking regulator,protein-coding,"Pax8(Paired,Homeobox)",1,-134,chr13,13587701,13592850
1927,TFF1,Kazn,"kazrin, periplakin interacting protein",protein-coding,"Pax8(Paired,Homeobox)",1,-149,chr4,142236351,142242750



  AP1

GFP: 7517 peak×motif rows | 2514 unique genes


...

,Sample,Gene Name,Gene Description,Gene Type,Motif,Motif Hits,Distance to TSS,Chr,Start,End
0,TFF1,Plut,"PDX1 associated lncRNA, upregulator of transcr...",ncRNA,Fosl2(bZIP),2,21,chr5,147267851,147275150
1,TFF1,Mir7039,microRNA 7039,ncRNA,Fosl2(bZIP),8,-709,chr5,144895401,144899650
2,TFF1,Lrrc38,leucine rich repeat containing 38,protein-coding,Fosl2(bZIP),4,-725,chr4,143344551,143353500
3,TFF1,St8sia1,"ST8 alpha-N-acetyl-neuraminide alpha-2,8-sialy...",protein-coding,Fosl2(bZIP),2,-623,chr6,142963251,142966900
4,TFF1,Grifin,galectin-related inter-fiber protein,protein-coding,Fosl2(bZIP),2,-408,chr5,140561951,140569000
...,...,...,...,...,...,...,...,...,...,...
5834,TFF1,Kazn,"kazrin, periplakin interacting protein",protein-coding,AP-1(bZIP),2,-149,chr4,142236351,142242750
5835,TFF1,Hira,histone cell cycle regulator,protein-coding,AP-1(bZIP),1,675,chr16,18875301,18879550
5836,TFF1,Snord100,"small nucleolar RNA, C/D box 100",snoRNA,AP-1(bZIP),5,-454,chr10,23782651,23789900
5837,TFF1,Marf1,meiosis regulator and mRNA stability 1,protein-coding,AP-1(bZIP),1,399,chr16,14157051,14160700



  p53

GFP: 3474 peak×motif rows | 1919 unique genes


,Sample,Gene Name,Gene Description,Gene Type,Motif,Motif Hits,Distance to TSS,Chr,Start,End
0,GFP,Mir1981,microRNA 1981,ncRNA,p63(p53),4,-862,chr1,184821801,184824900
1,GFP,Cyp24a1,"cytochrome P450, family 24, subfamily a, polyp...",protein-coding,p63(p53),2,-655,chr2,170496651,170498950
2,GFP,Bpifb1,"BPI fold containing family B, member 1",protein-coding,p63(p53),3,95,chr2,154197951,154203000
3,GFP,Fgf3,fibroblast growth factor 3,protein-coding,p63(p53),2,-987,chr7,144834451,144840800
4,GFP,Tas1r2,"taste receptor, type 1, member 2",protein-coding,p63(p53),3,-863,chr4,139647551,139657800
...,...,...,...,...,...,...,...,...,...,...
3469,GFP,Dda1,DET1 and DDB1 associated 1,protein-coding,p73(p53),2,-994,chr8,71456751,71479650
3470,GFP,Tmem53,transmembrane protein 53,protein-coding,p73(p53),1,210,chr4,117249601,117255500
3471,GFP,Acot11,acyl-CoA thioesterase 11,protein-coding,p73(p53),1,-43,chr4,106801901,106808200
3472,GFP,Fendrr,Foxf1 adjacent non-coding developmental regula...,ncRNA,p73(p53),1,-140,chr8,121078601,121087900



TFF1: 3856 peak×motif rows | 2046 unique genes


,Sample,Gene Name,Gene Description,Gene Type,Motif,Motif Hits,Distance to TSS,Chr,Start,End
0,TFF1,Bhlhe23,"basic helix-loop-helix family, member e23",protein-coding,p63(p53),2,-175,chr2,180775001,180779150
1,TFF1,Fam209,family with sequence similarity 209,protein-coding,p63(p53),1,-304,chr2,172470701,172473800
2,TFF1,Kcns1,"K+ voltage-gated channel, subfamily S, 1",protein-coding,p63(p53),2,-162,chr2,164170001,164172550
3,TFF1,Mmp24,matrix metallopeptidase 24,protein-coding,p63(p53),1,-892,chr2,155772401,155776500
4,TFF1,Gm13133,predicted gene 13133,ncRNA,p63(p53),2,-907,chr4,154546501,154552250
...,...,...,...,...,...,...,...,...,...,...
3851,TFF1,Ptgs2os,"prostaglandin-endoperoxide synthase 2, opposit...",ncRNA,p53(p53),2,199,chr1,150097101,150102250
3852,TFF1,Serp2,stress-associated endoplasmic reticulum protei...,protein-coding,p53(p53),1,-834,chr14,76554801,76560700
3853,TFF1,Lamb3,"laminin, beta 3",protein-coding,p53(p53),2,-119,chr1,193299351,193304400
3854,TFF1,Mns1,meiosis-specific nuclear structural protein 1,protein-coding,p53(p53),1,-829,chr9,72434951,72440450


## 7. GFP vs TFF1 comparison


In [27]:
comp_tables = {}

for fam in TF_FAMILIES:
    g_gfp  = set(
        tf_gene_tables[fam].get("GFP",  pd.DataFrame())
        .get("Gene Name", pd.Series()).dropna()
    )
    g_tff1 = set(
        tf_gene_tables[fam].get("TFF1", pd.DataFrame())
        .get("Gene Name", pd.Series()).dropna()
    )

    shared    = g_gfp & g_tff1
    gfp_only  = g_gfp - g_tff1
    tff1_only = g_tff1 - g_gfp

    print(f"\n{'='*70}")
    print(f"  {fam}  —  "
          f"Shared: {len(shared)}  |  GFP only: {len(gfp_only)}  |  TFF1 only: {len(tff1_only)}")
    print(f"{'='*70}")

    rows = (
          [{"Gene Name": g, "In GFP_H3K4me3": True,  "In TFF1_H3K4me3": True,  "Category": "Shared"}    for g in sorted(shared)]
        + [{"Gene Name": g, "In GFP_H3K4me3": True,  "In TFF1_H3K4me3": False, "Category": "GFP only"}  for g in sorted(gfp_only)]
        + [{"Gene Name": g, "In GFP_H3K4me3": False, "In TFF1_H3K4me3": True,  "Category": "TFF1 only"} for g in sorted(tff1_only)]
    )
    comp = pd.DataFrame(rows).sort_values("Category").reset_index(drop=True)
    comp_tables[fam] = comp
    display(comp)


  Pax  —  Shared: 715  |  GFP only: 1676  |  TFF1 only: 1200


,Gene Name,In GFP_H3K4me3,In TFF1_H3K4me3,Category
0,Ppp1r26,True,False,GFP only
1,Dusp18,True,False,GFP only
2,Dtwd2,True,False,GFP only
3,Dstyk,True,False,GFP only
4,Dsel,True,False,GFP only
...,...,...,...,...
3586,Gm15471,False,True,TFF1 only
3587,Gm15420,False,True,TFF1 only
3588,Gm15222,False,True,TFF1 only
3589,Gm14167,False,True,TFF1 only



  AP1  —  Shared: 975  |  GFP only: 1539  |  TFF1 only: 1765


,Gene Name,In GFP_H3K4me3,In TFF1_H3K4me3,Category
0,Slc25a14,True,False,GFP only
1,Aagab,True,False,GFP only
2,AY074887,True,False,GFP only
3,AU020206,True,False,GFP only
4,A930006K02Rik,True,False,GFP only
...,...,...,...,...
4274,Gm10510,False,True,TFF1 only
4275,Gm1043,False,True,TFF1 only
4276,Gm10369,False,True,TFF1 only
4277,Gm13842,False,True,TFF1 only



  p53  —  Shared: 748  |  GFP only: 1171  |  TFF1 only: 1298


,Gene Name,In GFP_H3K4me3,In TFF1_H3K4me3,Category
0,Senp2,True,False,GFP only
1,A230077H06Rik,True,False,GFP only
2,9530068E07Rik,True,False,GFP only
3,9530034E10Rik,True,False,GFP only
4,9330162012Rik,True,False,GFP only
...,...,...,...,...
3212,Gm19605,False,True,TFF1 only
3213,Gm16897,False,True,TFF1 only
3214,Gm16861,False,True,TFF1 only
3215,Gm38882,False,True,TFF1 only


## 8. Export


In [28]:
for fam in TF_FAMILIES:
    for label, tbl in tf_gene_tables[fam].items():
        if not tbl.empty:
            out = OUTPUT_DIR / f"{fam}_{label}_H3K4me3_promoter_genes.tsv"
            tbl.to_csv(out, sep="\t", index=False)
            print(f"Saved: {out}")

for fam, comp in comp_tables.items():
    if not comp.empty:
        out = OUTPUT_DIR / f"{fam}_GFP_vs_TFF1_comparison.tsv"
        comp.to_csv(out, sep="\t", index=False)
        print(f"Saved: {out}")

out = OUTPUT_DIR / "H3K4me3_significant_motifs.tsv"
sig_df.to_csv(out, sep="\t", index=False)
print(f"Saved: {out}")

Saved: /coh_labs/mvandenbrink/users/pkaur/6_tff1/3_cutnrun_relaxed/6_homer/analysis_output/Pax_GFP_H3K4me3_promoter_genes.tsv
Saved: /coh_labs/mvandenbrink/users/pkaur/6_tff1/3_cutnrun_relaxed/6_homer/analysis_output/Pax_TFF1_H3K4me3_promoter_genes.tsv
Saved: /coh_labs/mvandenbrink/users/pkaur/6_tff1/3_cutnrun_relaxed/6_homer/analysis_output/AP1_GFP_H3K4me3_promoter_genes.tsv
Saved: /coh_labs/mvandenbrink/users/pkaur/6_tff1/3_cutnrun_relaxed/6_homer/analysis_output/AP1_TFF1_H3K4me3_promoter_genes.tsv
Saved: /coh_labs/mvandenbrink/users/pkaur/6_tff1/3_cutnrun_relaxed/6_homer/analysis_output/p53_GFP_H3K4me3_promoter_genes.tsv
Saved: /coh_labs/mvandenbrink/users/pkaur/6_tff1/3_cutnrun_relaxed/6_homer/analysis_output/p53_TFF1_H3K4me3_promoter_genes.tsv
Saved: /coh_labs/mvandenbrink/users/pkaur/6_tff1/3_cutnrun_relaxed/6_homer/analysis_output/Pax_GFP_vs_TFF1_comparison.tsv
Saved: /coh_labs/mvandenbrink/users/pkaur/6_tff1/3_cutnrun_relaxed/6_homer/analysis_output/AP1_GFP_vs_TFF1_comparison.t